## Notebook to create datasets 
Saves CSV files in the /data directory.

In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import sys
sys.path.append("src")

import torch
import gc
import random
import pandas as pd

import _dataset
import _prompt
import _mapping

## Save stepwise reasoning prompts

In [16]:
random.seed(42)

add_ds = _dataset.create_h_dataset(num_digits=3, num_samples=256)

In [18]:
def save_prompts(add_ds, get_prompt_fn, filename, divide_num=100):

    # Create a list to store prompt data
    prompt_data = []

    for add_ds_entry in add_ds:
        base_1_digits = add_ds_entry["base_1_digits"]
        base_2_digits = add_ds_entry["base_2_digits"]
        base_1_num = add_ds_entry["base_1_num"]
        base_2_num = add_ds_entry["base_2_num"]
        base_sum = add_ds_entry["base_sum"]
        source_1_digits = add_ds_entry["source_1_digits"]
        source_2_digits = add_ds_entry["source_2_digits"]
        source_1_num = add_ds_entry["source_1_num"]
        source_2_num = add_ds_entry["source_2_num"]
        source_sum = add_ds_entry["source_sum"]

        base_prompt = get_prompt_fn(base_1_digits, base_2_digits, base_1_num, base_2_num)
        truncated_base_prompt = _prompt.divide_prompt(divide_num, base_prompt)[0]
        source_prompt = get_prompt_fn(source_1_digits, source_2_digits, source_1_num, source_2_num)
        truncated_source_prompt = _prompt.divide_prompt(divide_num, source_prompt)[0]

        # Add prompt data to list
        prompt_data.append({
            'base_1_digits': base_1_digits,
            'base_2_digits': base_2_digits,
            'base_1_num': base_1_num,
            'base_2_num': base_2_num,
            'base_sum': base_sum,
            'source_1_digits': source_1_digits,
            'source_2_digits': source_2_digits,
            'source_1_num': source_1_num,
            'source_2_num': source_2_num,
            'source_sum': source_sum,
            'base_prompt': truncated_base_prompt,
            'source_prompt': truncated_source_prompt
        })

    # Save to CSV
    prompt_df = pd.DataFrame(prompt_data)
    prompt_df.to_csv(f'../data/{filename}', index=False)
    print(f"Saved {len(prompt_data)} prompts to {filename}")

In [21]:
# save_prompts(add_ds, _prompt.get_stepwise_prompt, "hundreds_three_digit_sums_prompts.csv")
# save_prompts(add_ds, _prompt.get_stepwise_prompt, "hundreds_three_digit_sums_pre_result_prompts.csv", divide_num=26)
save_prompts(add_ds, _prompt.get_R1_prompt, "R1/h_prompts_pre_result.csv", divide_num=14)

Saved 256 prompts to R1/h_prompts_pre_result.csv


## Divide prompts at intervention locations

In [ ]:
def divide_prompts(prompt_data, intervention_ids_dict=_mapping.intervene_ids_stepwise_3_digit, get_counterfactual_sum_fn=_prompt.get_counterfactual_sum_OSS_3_digit):

    divided_prompts = []

    for index, row in prompt_data.iterrows():
        base_prompt = row['base_prompt']
        source_prompt = row['source_prompt']
        
        # Process base prompt
        for i in intervention_ids_dict["all"]:
            base_before, base_number, base_after = _prompt.divide_prompt(i, base_prompt)
            source_before, source_number, source_after = _prompt.divide_prompt(i, source_prompt)
            if base_after == "" or source_after == "":
                break
            counterfactual_sum = get_counterfactual_sum_fn(i, **row)
            
            divided_prompts.append({
                'base_1_digits': row['base_1_digits'],
                'base_2_digits': row['base_2_digits'],
                'base_1_num': row['base_1_num'],
                'base_2_num': row['base_2_num'],
                'base_sum': row['base_sum'],
                'source_1_digits': row['source_1_digits'],
                'source_2_digits': row['source_2_digits'],
                'source_1_num': row['source_1_num'],
                'source_2_num': row['source_2_num'],
                'source_sum': row['source_sum'],
                'intervention_id': i,
                'base_before': base_before,
                'base_number': base_number,
                'base_after': base_after,
                'source_before': source_before,
                'source_number': source_number,
                'source_after': source_after,
                'counterfactual_sum': counterfactual_sum,
                'in_restatement': i in intervention_ids_dict['restatement'],
                'in_reasoning': i in intervention_ids_dict['reasoning'],
                'in_result': i in intervention_ids_dict['result'],
                # 'in_copy': i in _mapping.intervention_ids_dict['copy'],
                # 'in_intermediate_sums': i in _mapping.intervention_ids_dict['intermediate_sums'],
            })

    return pd.DataFrame(divided_prompts)


In [22]:
# Divide prompts at intervention locations
prompt_df = pd.read_csv('../data/R1/h_prompts_pre_result.csv')
divided_df = divide_prompts(prompt_df, _mapping.intervene_ids_R1_3_digit_h, _prompt.get_counterfactual_sum_R1_3_digit)

# Save divided prompts to CSV
divided_df.to_csv('../data/R1/h_divided_prompts_pre_result.csv', index=False)
print(f"Saved {len(divided_df)} divided prompts to data/R1/divided_prompts.csv")

Saved 768 divided prompts to data/R1/divided_prompts.csv
